# C1 Sensor Data Exploration

This notebook profiles the locally downloaded dataset before implementing the C1 validation service.

The dataset must be placed in `data/raw/`. It is intentionally excluded from Git. The final year is reserved exclusively for testing; this notebook only inspects the data and demonstrates the chronological split.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# The split lives in the C1 service. Make it importable when the notebook
# runs from the repository root or from notebooks/ without an installed workspace.
SERVICE_SRC = Path("services/c1-sensor-quality/src").resolve()
if not SERVICE_SRC.is_dir():
    SERVICE_SRC = Path("../services/c1-sensor-quality/src").resolve()
if str(SERVICE_SRC) not in sys.path:
    sys.path.insert(0, str(SERVICE_SRC))

from c1_sensor_quality.split import chronological_split

data_dirs = [Path("data/raw"), Path("../data/raw")]
candidates = sorted(
    path
    for data_dir in data_dirs
    for path in [*data_dir.glob("*.csv"), *data_dir.glob("*.xlsx"), *data_dir.glob("*.xls")]
)

if not candidates:
    raise FileNotFoundError(
        "No CSV or Excel file found in data/raw. Download the dataset there before running this notebook."
    )

DATA_PATH = candidates[0]
print(f"Using: {DATA_PATH}")

In [3]:
if DATA_PATH.suffix.lower() == ".csv":
    df = pd.read_csv(DATA_PATH)
else:
    df = pd.read_excel(DATA_PATH)

df.head()

,timestamp,machine_id,operating_mode,temperature,vibration,pressure,rpm,power,anomaly,anomaly_type,failure_probability_gt,failure_in_7_days,remaining_useful_life_hours,failure_type,important_features,maintenance_action,priority,concept_drift_active
0,"2012-01-14 00:00:00,M001,HIGH_LOAD,46.07525961...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"2012-01-14 00:00:00,M002,NORMAL,46.58425922252...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"2012-01-14 00:00:00,M003,IDLE,40.5464513381037...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2012-01-14 00:00:00,M004,NORMAL,48.470209,2.953438,97.447963,1515.488150,211.998125,False,NaN,0.374622,False,625.0,cooling_failure,temperature,repair_cooling_system,low,False
4,2012-01-14 00:00:00,M005,NORMAL,39.141233,1.222667,99.561297,1504.955537,209.234533,False,NaN,0.049056,False,950.0,cooling_failure,temperature,repair_cooling_system,low,False


## Basic profile

In [4]:
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
display(df.dtypes.to_frame("dtype"))
display(df.describe(include="all").T)

Rows: 525,600
Columns: 18


,dtype
timestamp,str
machine_id,str
operating_mode,str
temperature,float64
vibration,float64
pressure,float64
rpm,float64
power,float64
anomaly,object
anomaly_type,str


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
timestamp,525600,396665,2012-01-14 00:00:00,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
machine_id,155215,6,M004,26280,NaN,NaN,NaN,NaN,NaN,NaN,NaN
operating_mode,155215,4,NORMAL,92861,NaN,NaN,NaN,NaN,NaN,NaN,NaN
temperature,154888.0,NaN,NaN,NaN,59.050962,45.713095,26.531695,48.949295,57.106129,64.866697,1091.799618
vibration,155215.0,NaN,NaN,NaN,4.065706,1.73495,0.484968,2.641259,3.971989,5.287272,9.512809
pressure,155215.0,NaN,NaN,NaN,96.927726,1.761143,91.552289,95.557489,97.02254,98.357779,101.505433
rpm,155215.0,NaN,NaN,NaN,1773.750162,597.45603,407.882312,1500.157132,1648.977456,2194.356624,3600.761871
power,155215.0,NaN,NaN,NaN,251.01593,91.728995,38.973827,212.873813,230.370947,330.90749,513.360336
anomaly,155215,2,False,154256,NaN,NaN,NaN,NaN,NaN,NaN,NaN
anomaly_type,959,3,missing_value,327,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
df["anomaly"].value_counts(dropna=False)

anomaly
NaN      370385
False    154256
True        959
Name: count, dtype: int64

In [5]:
missing = (
    df.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_rate=lambda table: table["missing_count"] / len(df))
    .sort_values("missing_count", ascending=False)
)
display(missing)

,missing_count,missing_rate
anomaly_type,524641,0.998175
temperature,370712,0.705312
failure_probability_gt,370385,0.704690
machine_id,370385,0.704690
vibration,370385,0.704690
operating_mode,370385,0.704690
pressure,370385,0.704690
rpm,370385,0.704690
power,370385,0.704690
anomaly,370385,0.704690


## Data quality and coverage

In [6]:
required_columns = {
    "timestamp",
    "machine_id",
    "operating_mode",
    "temperature",
    "vibration",
    "pressure",
    "rpm",
    "power",
}
print("Missing required columns:", sorted(required_columns - set(df.columns)))
print("Duplicate rows:", int(df.duplicated().sum()))

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
print("Invalid timestamps:", int(df["timestamp"].isna().sum()))
print("Time range:", df["timestamp"].min(), "to", df["timestamp"].max())

Missing required columns: []
Duplicate rows: 0


/tmp/ipykernel_157550/3995980870.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")


Invalid timestamps: 370385
Time range: 2012-01-14 00:00:00 to 2015-01-12 23:00:00


In [7]:
for column in [
    "machine_id",
    "operating_mode",
    "anomaly_type",
    "failure_type",
    "maintenance_action",
    "priority",
]:
    if column in df:
        print(f"\n{column}")
        display(df[column].value_counts(dropna=False).to_frame("count"))


machine_id


,count
machine_id,
NaN,370385
M004,26280
M010,26280
M006,26266
M005,26259
M013,26216
M014,23914



operating_mode


,count
operating_mode,
NaN,370385
NORMAL,92861
HIGH_LOAD,39043
IDLE,15516
OVERLOAD,7795



anomaly_type


,count
anomaly_type,
NaN,524641
missing_value,327
sensor_frozen,326
sensor_spike,306


... (2 outputs left out)


maintenance_action


,count
maintenance_action,
NaN,370385
repair_cooling_system,155215



priority


,count
priority,
NaN,370385
low,84113
medium,55911
high,15191


## Chronological split

The first two years are available for training and validation. The final year is held out for testing and must not be used for model selection.

In [ ]:
valid_df = df.dropna(subset=["timestamp"]).reset_index(drop=True)

train, validation, test = chronological_split(valid_df)

split_summary = pd.DataFrame(
    {
        "partition": ["train", "validation", "test"],
        "start": [part["timestamp"].min() for part in (train, validation, test)],
        "end": [part["timestamp"].max() for part in (train, validation, test)],
        "rows": [len(train), len(validation), len(test)],
    }
)
display(split_summary)
assert train["timestamp"].max() < validation["timestamp"].min()
assert validation["timestamp"].max() < test["timestamp"].min()

## Next steps

- Confirm the actual column names against the assignment dataset.
- Decide which sensor ranges and categorical values are valid.
- Document anomalies and missing-value patterns.
- Implement the validated schema and processing rules in the C1 service.
